# Data Cleaning & Preprocessing — Healthcare Analytics Project

This notebook cleans the four raw datasets (`patients`, `appointments`, `treatments`, `billing`), engineers features needed for analysis (`age_group`, `cost_category`, `bill_category`), merges them into a single analytical table, and saves the result as `healthcare_master.csv` — the file used by `sql/healthcare_queries.sql` and `notebook/ML_work.ipynb`.

> **Note:** `doctors.csv` is loaded and explored separately for context (doctor specialization / workload) but is not part of the master table, since `doctor_id` is already present in `appointments`.


## 0. Setup


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## 1. Patients Dataset


In [ ]:
patients = pd.read_csv("data/raw_data/patients.csv")

print(patients.shape)
patients.head()


In [ ]:
# Null and duplicate check
print(patients.isnull().sum())
print('Duplicates:', patients.duplicated().sum())

patients.drop_duplicates(inplace=True)


In [ ]:
# Convert date columns
patients['date_of_birth'] = pd.to_datetime(patients['date_of_birth'])
patients['registration_date'] = pd.to_datetime(patients['registration_date'])

# Engineer age and age_group
patients['age'] = 2026 - patients['date_of_birth'].dt.year

def age_group(age):
    if age < 18:
        return 'Child'
    elif age < 40:
        return 'Young Adult'
    elif age < 60:
        return 'Middle Age'
    else:
        return 'Senior'

patients['age_group'] = patients['age'].apply(age_group)
patients['age_group'].value_counts()


In [ ]:
sns.countplot(x='age_group', data=patients,
              order=patients['age_group'].value_counts().index)
plt.title('Patients by Age Group')
plt.show()


## 2. Appointments Dataset


In [ ]:
appointments = pd.read_csv("data/raw_data/appointments.csv")

print(appointments.shape)
appointments.head()


In [ ]:
print(appointments.isnull().sum())
print('Duplicates:', appointments.duplicated().sum())

appointments.drop_duplicates(inplace=True)
appointments['appointment_date'] = pd.to_datetime(appointments['appointment_date'])


In [ ]:
# Appointment status distribution
appointments['status'].value_counts()


In [ ]:
sns.countplot(x='status', data=appointments)
plt.title('Appointment Status Distribution')
plt.show()


## 3. Treatments Dataset


In [ ]:
treatments = pd.read_csv("data/raw_data/treatments.csv")

print(treatments.shape)
treatments.head()


In [ ]:
print(treatments.isnull().sum())
print('Duplicates:', treatments.duplicated().sum())

treatments.drop_duplicates(inplace=True)
treatments['treatment_date'] = pd.to_datetime(treatments['treatment_date'])


In [ ]:
# Engineer cost_category
def cost_category(cost):
    if cost < 5000:
        return 'Low Cost'
    elif cost < 20000:
        return 'Medium Cost'
    else:
        return 'High Cost'

treatments['cost_category'] = treatments['cost'].apply(cost_category)
treatments[['cost', 'cost_category']].head()


## 4. Billing Dataset


In [ ]:
billing = pd.read_csv("data/raw_data/billing.csv")

print(billing.shape)
billing.head()


In [ ]:
print(billing.isnull().sum())
print('Duplicates:', billing.duplicated().sum())

billing.drop_duplicates(inplace=True)
billing['bill_date'] = pd.to_datetime(billing['bill_date'])


In [ ]:
# Engineer bill_category (mirrors cost_category logic for billing amounts)
def bill_category(amount):
    if amount < 5000:
        return 'Low'
    elif amount < 20000:
        return 'Medium'
    else:
        return 'High'

billing['bill_category'] = billing['amount'].apply(bill_category)
billing['payment_status'].value_counts()


## 5. Doctors Dataset (context only — not merged into master table)


In [ ]:
doctors = pd.read_csv("data/raw_data/doctors.csv")
doctors.drop_duplicates(inplace=True)

print(doctors.shape)
doctors['specialization'].value_counts()


## 6. Merge Into a Single Master Table

`patients` → `appointments` (on `patient_id`) → `treatments` (on `appointment_id`) → `billing` (on `patient_id`).

**⚠️ Important:** Verify these join keys (`patient_id`, `appointment_id`) match the actual column names in your raw CSVs before running this — adjust if your files use different key names.


In [ ]:
master = patients.merge(appointments, on='patient_id', how='inner')

master = master.merge(
    treatments,
    on='appointment_id',
    how='left'
)

master = master.merge(
    billing,
    on='patient_id',
    how='left'
)

print('Master shape:', master.shape)
master.head()


In [ ]:
# Clean up duplicate / pandas-generated suffix columns from the merges (e.g. patient_id_x, patient_id_y)
# Inspect first, then drop or rename as needed so the final table has one clean column per field.
print([c for c in master.columns if c.endswith('_x') or c.endswith('_y')])


In [ ]:
# Example cleanup once you've checked the duplicate columns above:
# master.rename(columns={'patient_id_x': 'patient_id'}, inplace=True)
# master.drop(columns=['patient_id_y'], inplace=True)


## 7. Sanity Checks on the Merged Table


In [ ]:
print('Total rows:', len(master))
print('Null values per column:')
print(master.isnull().sum())


In [ ]:
no_show_rate = (master['status'] == 'No-show').mean() * 100
print(f'No-show rate: {no_show_rate:.1f}%')

total_revenue = master['amount'].sum()
print(f'Total revenue: ₹{total_revenue:,.2f}')


## 8. Save the Cleaned Master Dataset


In [ ]:
master.to_csv("data/cleaned/healthcare_master.csv", index=False)
print('Saved: data/cleaned/healthcare_master.csv')


---
**Next step:** open `notebook/ML_work.ipynb`, which loads `data/healthcare_master.csv` for no-show prediction, risk segmentation, and statistical validation.
